# Combine rerun HSC with CLASH 

This notebook combines your rerun HSC file with CLASH results, then selects the best variant per unique HSC function using minimum DL.

In [ ]:
import os
import csv
import numpy as np
from pathlib import Path

# Requested complexities
comps = [6, 7, 8, 9, 10]

# Base paths
base_root = "../results/HSC_CLASH"
funcs_root = f"{base_root}/funcs"
# hsc_funcs_root = f"{base_root}/results/HSC_funcs_next_500/funcs"
hsc_funcs_root = f"{base_root}/HSC_funcs"

# Templates
hsc_rerun_template = f"{base_root}/results/HSC_funcs/codelen_matches_comp{{comp}}_HSC_CLASH.dat"
clash_template = f"{base_root}/CLASH/codelen_matches_comp{{comp}}.dat"

# Output directory for per-comp + combined files
output_dir = f"{base_root}/results/HSC_funcs"
os.makedirs(output_dir, exist_ok=True)

print("Checking input files for requested comps...")
for comp in comps:
    hsc_rerun_path = hsc_rerun_template.format(comp=comp)
    clash_path = clash_template.format(comp=comp)
    all_fcn_path = f"{funcs_root}/compl_{comp}/all_equations_{comp}.txt"
    aifeyn_path = f"{funcs_root}/compl_{comp}/aifeyn_{comp}.txt"
    hsc_all_fcn_path = f"{hsc_funcs_root}/compl_{comp}/all_equations_{comp}.txt"
    hsc_unique_fcn_path = f"{hsc_funcs_root}/compl_{comp}/unique_equations_{comp}.txt"
    hsc_matches_candidate = f"{hsc_funcs_root}/compl_{comp}/matches_{comp}.txt"

    print(f"\ncomp {comp}")
    for p in [
        hsc_rerun_path, clash_path, all_fcn_path, aifeyn_path,
        hsc_all_fcn_path, hsc_unique_fcn_path, hsc_matches_candidate
    ]:
        print(f"  {'OK' if os.path.exists(p) else 'MISSING'}  {p}")

In [ ]:
def read_list(path):
    with open(path, 'r') as f:
        return [ln.strip() for ln in f if ln.strip()]

def read_codelen_matches(path, functions, matches_path=None):
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    data = np.genfromtxt(path)
    if data.size == 0:
        raise ValueError(f"Empty file: {path}")
    if data.ndim == 1:
        data = data.reshape(1, -1)

    max_params = 4
    negloglike = data[:, 0]
    codelen = data[:, 1]
    index = data[:, 2].astype(int)

    if matches_path is not None:
        matches = np.array([int(x.strip()) for x in open(matches_path) if x.strip()], dtype=int)
        if len(matches) != len(index):
            min_len = min(len(matches), len(index))
            data = data[:min_len]
            index = matches[:min_len]
            negloglike = data[:, 0]
            codelen = data[:, 1]
        else:
            index = matches

    # Align functions to the row data. If matches provide indices, subset the full list.
    if len(functions) == len(negloglike):
        row_functions = functions
    else:
        row_functions = [functions[i] for i in index if 0 <= i < len(functions)]
        if len(row_functions) != len(negloglike):
            min_len = min(len(row_functions), len(negloglike))
            row_functions = row_functions[:min_len]
            negloglike = negloglike[:min_len]
            codelen = codelen[:min_len]
            index = index[:min_len]
            data = data[:min_len]

    params = data[:, 3:3 + max_params]
    delta = data[:, 3 + max_params:3 + 2 * max_params]
    nconv = data[:, -3]
    niter = data[:, -2]
    time = data[:, -1]

    return {
        'negloglike': negloglike,
        'codelen': codelen,
        'index': index,
        'params': params,
        'delta': delta,
        'Nconv': nconv,
        'Niter': niter,
        'time': time,
        'functions': row_functions,
    }

# ... rest of cell unchanged

def combine_hsc_clash(hsc_data, clash_data, all_fcn, aifeyn):
    n_funcs = len(hsc_data['functions'])

    combined_negloglike = np.full(n_funcs, np.nan)
    combined_codelen = np.full(n_funcs, np.nan)
    combined_DL = np.full(n_funcs, np.nan)
    hsc_sum = np.full(n_funcs, np.nan)
    clash_sum = np.full(n_funcs, np.nan)
    hsc_negloglike = np.full(n_funcs, np.nan)
    hsc_codelen = np.full(n_funcs, np.nan)
    clash_negloglike = np.full(n_funcs, np.nan)
    clash_codelen = np.full(n_funcs, np.nan)
    combined_params = np.zeros((n_funcs, 4))
    combined_delta = np.zeros((n_funcs, 4))
    combined_Nconv = np.zeros(n_funcs)
    combined_Niter = np.zeros(n_funcs)
    combined_time = np.zeros(n_funcs)
    combined_index = hsc_data['index'].copy()
    valid_mask = np.zeros(n_funcs, dtype=bool)

    aifeyn_map = {f: aifeyn[i] for i, f in enumerate(all_fcn)}

    clash_map = {}
    for i, f in enumerate(clash_data['functions']):
        clash_map.setdefault(f, []).append(i)

    hsc_map = {f: i for i, f in enumerate(hsc_data['functions'])}
    shared_functions = sorted(set(hsc_map.keys()) & set(clash_map.keys()))
    print(f"Shared functions: {len(shared_functions)}")

    for fcn in shared_functions:
        hsc_i = hsc_map[fcn]
        idx = hsc_i
        print(len(hsc_data['negloglike']))
        print(hsc_i)

        hsc_neglog = hsc_data['negloglike'][hsc_i]
        hsc_code = hsc_data['codelen'][hsc_i]
        if not np.isfinite(hsc_neglog) or not np.isfinite(hsc_code):
            continue

        best_clash_i = None
        best_clash_dl = np.inf

        for clash_i in clash_map[fcn]:
            c_neg = clash_data['negloglike'][clash_i]
            c_cod = clash_data['codelen'][clash_i]
            if not np.isfinite(c_neg) or not np.isfinite(c_cod):
                continue
            c_dl = c_neg + c_cod
            if c_dl < best_clash_dl:
                best_clash_dl = c_dl
                best_clash_i = clash_i

        if best_clash_i is None:
            continue

        c_neg = clash_data['negloglike'][best_clash_i]
        c_cod = clash_data['codelen'][best_clash_i]

        combined_negloglike[idx] = hsc_neglog + c_neg
        combined_codelen[idx] = hsc_code + c_cod
        hsc_sum[idx] = hsc_neglog + hsc_code
        clash_sum[idx] = c_neg + c_cod
        hsc_negloglike[idx] = hsc_neglog
        hsc_codelen[idx] = hsc_code
        clash_negloglike[idx] = c_neg
        clash_codelen[idx] = c_cod

        aif_val = aifeyn_map.get(fcn, np.nan)
        combined_DL[idx] = combined_negloglike[idx] + combined_codelen[idx] + aif_val

        combined_params[idx] = hsc_data['params'][hsc_i]
        combined_delta[idx] = hsc_data['delta'][hsc_i]
        combined_Nconv[idx] = hsc_data['Nconv'][hsc_i]
        combined_Niter[idx] = hsc_data['Niter'][hsc_i]
        combined_time[idx] = hsc_data['time'][hsc_i]
        valid_mask[idx] = True

    return {
        'negloglike': combined_negloglike,
        'codelen': combined_codelen,
        'DL': combined_DL,
        'hsc_sum': hsc_sum,
        'clash_sum': clash_sum,
        'hsc_negloglike': hsc_negloglike,
        'hsc_codelen': hsc_codelen,
        'clash_negloglike': clash_negloglike,
        'clash_codelen': clash_codelen,
        'index': combined_index,
        'params': combined_params,
        'delta': combined_delta,
        'Nconv': combined_Nconv,
        'Niter': combined_Niter,
        'time': combined_time,
        'valid_mask': valid_mask,
    }



In [ ]:
best_by_comp = {}
summary_rows = []

for comp in comps:
    print("\n" + "=" * 70)
    print(f"Processing comp {comp}")
    print("=" * 70)

    hsc_rerun_path = hsc_rerun_template.format(comp=comp)
    clash_path = clash_template.format(comp=comp)
    all_fcn_path = f"{funcs_root}/compl_{comp}/all_equations_{comp}.txt"
    aifeyn_path = f"{funcs_root}/compl_{comp}/aifeyn_{comp}.txt"
    hsc_all_fcn_path = f"{hsc_funcs_root}/compl_{comp}/all_equations_{comp}.txt"
    hsc_unique_fcn_path = f"{hsc_funcs_root}/compl_{comp}/unique_equations_{comp}.txt"
    hsc_matches_candidate = f"{hsc_funcs_root}/compl_{comp}/matches_{comp}.txt"
    hsc_matches_path = hsc_matches_candidate if os.path.exists(hsc_matches_candidate) else None

    required = [hsc_rerun_path, clash_path, all_fcn_path, aifeyn_path, hsc_all_fcn_path, hsc_unique_fcn_path]
    missing = [p for p in required if not os.path.exists(p)]
    if missing:
        print(f"Skipping comp {comp} due to missing files:")
        for p in missing:
            print(" -", p)
        continue

    all_fcn = read_list(all_fcn_path)
    aifeyn = np.atleast_1d(np.genfromtxt(aifeyn_path))
    hsc_all_fcn = read_list(hsc_all_fcn_path)
    hsc_unique_fcn = read_list(hsc_unique_fcn_path)

    hsc_data = read_codelen_matches(hsc_rerun_path, hsc_all_fcn, matches_path=hsc_matches_path)
    clash_data = read_codelen_matches(clash_path, all_fcn)

    combined_data = combine_hsc_clash(hsc_data, clash_data, all_fcn, aifeyn)
    best_data = find_best_variants(hsc_unique_fcn, hsc_all_fcn, combined_data)

    n_valid = int(np.sum(combined_data['valid_mask']))
    n_best = sum(1 for fn in best_data['function'] if fn is not None)
    valid_unique_indices = np.unique(combined_data['index'][combined_data['valid_mask']])

    hsc_total = len(hsc_all_fcn)
    hsc_unique_total = len(hsc_unique_fcn)

    def normalize_fn(fn):
        return "".join(fn.split())

    clash_norm = {normalize_fn(fn): fn for fn in all_fcn}
    hsc_all_norm = {normalize_fn(fn): fn for fn in hsc_all_fcn}
    hsc_unique_norm = {normalize_fn(fn): fn for fn in hsc_unique_fcn}

    shared_all = set(hsc_all_norm.keys()) & set(clash_norm.keys())
    shared_unique = set(hsc_unique_norm.keys()) & set(clash_norm.keys())

    missing_hsc_unique = sorted(set(hsc_unique_norm.keys()) - set(clash_norm.keys()))

    print(f"HSC total funcs: {hsc_total}")
    print(f"HSC unique funcs: {hsc_unique_total}")
    print(f"Shared funcs (HSC all vs CLASH, normalized): {len(shared_all)}")
    print(f"Shared funcs (HSC unique vs CLASH, normalized): {len(shared_unique)}")
    print(f"Combined valid rows: {n_valid}")
    print(f"Best unique functions: {n_best}")
    print(f"Valid unique index count: {len(valid_unique_indices)}")
    print("Using matches file:", hsc_matches_path if hsc_matches_path is not None else "None")
    if missing_hsc_unique:
        preview = [hsc_unique_norm[k] for k in missing_hsc_unique[:5]]
        print(f"Missing HSC unique functions in CLASH (first 5): {preview}")

    output_codelen = f"{output_dir}/codelen_matches_comp{comp}_combined_from_rerun.dat"
    output_detailed = f"{output_dir}/detailed_results_comp{comp}_combined_from_rerun.dat"
    output_functions = f"{output_dir}/{comp}_CLASH_funcs_in_HSC_from_rerun.txt"

    write_output(output_codelen, best_data)
    write_detailed(output_detailed, best_data)
    write_functions(output_functions, best_data)

    print('Saved outputs:')
    print(' -', output_codelen)
    print(' -', output_detailed)
    print(' -', output_functions)

    best_by_comp[comp] = {
        'best_data': best_data,
        'n_valid': n_valid,
        'n_best': n_best,
        'hsc_total': hsc_total,
        'hsc_unique_total': hsc_unique_total,
        'shared_all': len(shared_all),
        'shared_unique': len(shared_unique),
    }
    summary_rows.append((comp, hsc_total, hsc_unique_total, len(shared_unique), n_best))

print("\nFinished comps:", sorted(best_by_comp.keys()))

In [ ]:
combined_records = {}

for comp, comp_payload in best_by_comp.items():
    best_data = comp_payload['best_data']
    for i, fn in enumerate(best_data['function']):
        if fn is None:
            continue
        dl = best_data['DL'][i]
        if not np.isfinite(dl):
            continue

        rec = {
            'function': fn,
            'comp': comp,
            'DL': float(dl),
            'HSC_negloglike': float(best_data['hsc_negloglike'][i]),
            'HSC_codelen': float(best_data['hsc_codelen'][i]),
            'CLASH_negloglike': float(best_data['clash_negloglike'][i]),
            'CLASH_codelen': float(best_data['clash_codelen'][i]),
            'Total_negloglike': float(best_data['negloglike'][i]),
            'Total_codelen': float(best_data['codelen'][i]),
            'AIFeyn': float(best_data['aifeyn'][i]),
            'HSC_sum': float(best_data['hsc_negloglike'][i] + best_data['hsc_codelen'][i]),
            'CLASH_sum': float(best_data['clash_negloglike'][i] + best_data['clash_codelen'][i]),
        }

        if fn not in combined_records or rec['DL'] < combined_records[fn]['DL']:
            combined_records[fn] = rec

try:
    import pandas as pd
    comp_summary_df = pd.DataFrame(summary_rows, columns=['comp', 'combined_valid_rows', 'best_unique_functions'])
    display(comp_summary_df.sort_values('comp').reset_index(drop=True))
except Exception:
    print('Per-comp summary:')
    for row in sorted(summary_rows, key=lambda x: x[0]):
        print(row)

combined_all_df = None
if len(combined_records) > 0:
    try:
        import pandas as pd
        combined_all_df = pd.DataFrame(list(combined_records.values())).sort_values('DL').reset_index(drop=True)
        # combined_all_df.insert(0, 'rank', np.arange(1, len(combined_all_df) + 1))

        combined_csv = f"{output_dir}/combined_selected_comps_5_7_8_9_10_fixed.csv"
        combined_txt = f"{output_dir}/combined_selected_comps_5_7_8_9_10_fixed.txt"
        combined_all_df.to_csv(combined_csv, index=False)
        combined_all_df.to_string(combined_txt, index=False)

        print(f"Combined unique functions across comps: {len(combined_all_df)}")
        print('Saved combined outputs:')
        print(' -', combined_csv)
        print(' -', combined_txt)
    except Exception as e:
        print('Could not build pandas combined table:', e)
else:
    print('No combined records found across requested comps.')

In [ ]:
# Final table from all requested complexities combined
top_n = 50

if 'combined_all_df' in globals() and combined_all_df is not None and len(combined_all_df) > 0:
    print(f"Showing {min(top_n, len(combined_all_df))} functions from combined all-comps table")
    display(combined_all_df.head(top_n))
else:
    print("combined_all_df is empty. Run Cell 4 then Cell 5 first.")

In [ ]:
from pathlib import Path
import pandas as pd

paths = [
    Path("../results/HSC_CLASH/results/HSC_funcs/combined_selected_comps_5_7_8_9_10_fixed.txt"),
    Path("../results/HSC_CLASH/results/HSC_funcs_next_500/combined_selected_comps_5_7_8_9_10_fixed.txt"),
]

out_txt = Path("../results/HSC_CLASH/results/HSC_funcs/combined_selected_comps_5_7_8_9_10_fixed_sorted_merged.txt")
out_csv = out_txt.with_suffix(".csv")
out_best100_txt = Path("../results/HSC_CLASH/results/HSC_funcs/combined_selected_comps_5_7_8_9_10_best_100_funcs.txt")

cols = [
    "function", "comp", "DL", "HSC_negloglike", "HSC_codelen",
    "CLASH_negloglike", "CLASH_codelen", "Total_negloglike",
    "Total_codelen", "AIFeyn", "HSC_sum", "CLASH_sum",
]


def read_selected_table(path):
    rows = []
    with path.open("r") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip() or line.lstrip().startswith("function"):
                continue
            parts = line.split()
            if len(parts) < 12:
                continue
            fn = " ".join(parts[:-11]).strip()
            tail = parts[-11:]
            try:
                rows.append({
                    "function": fn,
                    "comp": int(float(tail[0])),
                    "DL": float(tail[1]),
                    "HSC_negloglike": float(tail[2]),
                    "HSC_codelen": float(tail[3]),
                    "CLASH_negloglike": float(tail[4]),
                    "CLASH_codelen": float(tail[5]),
                    "Total_negloglike": float(tail[6]),
                    "Total_codelen": float(tail[7]),
                    "AIFeyn": float(tail[8]),
                    "HSC_sum": float(tail[9]),
                    "CLASH_sum": float(tail[10]),
                    "source": path.parent.name,
                })
            except ValueError:
                continue
    return pd.DataFrame(rows)

frames = [read_selected_table(p) for p in paths]
merged = pd.concat(frames, ignore_index=True)
merged = merged.dropna(subset=["function", "DL"])
merged = merged.sort_values(["function", "DL", "comp"], ascending=[True, True, True])
merged = merged.drop_duplicates(subset=["function"], keep="first")
merged = merged.sort_values("DL", ascending=True).reset_index(drop=True)

out_txt.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(out_csv, index=False)
merged.to_csv(out_txt, sep="\t", index=False, float_format="%.6f")

best_100_funcs = merged.head(100)["function"].tolist()
out_best100_txt.write_text("\n".join(best_100_funcs) + ("\n" if best_100_funcs else ""))

print(f"Merged {len(frames[0])} + {len(frames[1])} rows -> {len(merged)} unique functions")
print(f"Saved: {out_txt}")
print(f"Saved: {out_csv}")
print(f"Saved best 100 funcs: {out_best100_txt}")
display(merged.head(25))

In [ ]:
from pathlib import Path
import sympy as sp

# Build a compact LaTeX table with the requested formatting.
# It keeps the top 10 rows sorted by DL and uses formatted function strings.
latex_df = merged.loc[:, [
    "function",
    "comp",
    "Total_negloglike",
    "Total_codelen",
    "HSC_negloglike",
    "HSC_codelen",
    "CLASH_negloglike",
    "CLASH_codelen",
]].head(10).copy().reset_index(drop=True)
latex_df.insert(0, "Rank", range(1, len(latex_df) + 1))

latex_output = Path("../results/HSC_CLASH/results/HSC_funcs/combined_selected_comps_5_7_8_9_10_fixed_sorted_merged_top10_formatted.tex")


def format_function_for_latex(fn: str) -> str:
    # Map ESR symbols to presentation symbols.
    locals_map = {"x": sp.Symbol("r")}
    for i in range(20):
        locals_map[f"a{i}"] = sp.Symbol(rf"\\theta_{{{i}}}")

    try:
        expr = sp.sympify(fn, locals=locals_map, evaluate=True)
        return f"${sp.latex(expr)}$"
    except Exception:
        # Fallback: simple readable transform if parsing fails.
        s = str(fn)
        s = s.replace("x", "r")
        for i in range(20):
            s = s.replace(f"a{i}", rf"\\theta_{{{i}}}")
        return f"${s}$"


lines = [
    "\\begin{table*}[htbp]",
    "    \\centering",
    "    \\scriptsize",
    "    \\begin{tabular}{c c c c c c c c c}",
    "        \\hline",
    "        Rank & Function & Complexity & Total Residual$^{1}$ & Total Parameter$^{2}$ & HSC Residual & HSC Parameter & CLASH Residual & CLASH Parameter \\\\",
    "        \\hline",
]

for _, row in latex_df.iterrows():
    fn = format_function_for_latex(row["function"])
    lines.append(
        f"        {int(row['Rank'])} & {fn} & {int(row['comp'])} & "
        f"{row['Total_negloglike']:.2f} & {row['Total_codelen']:.2f} & "
        f"{row['HSC_negloglike']:.2f} & {row['HSC_codelen']:.2f} & "
        f"{row['CLASH_negloglike']:.2f} & {row['CLASH_codelen']:.2f} \\\\"
    )

lines += [
    "        \\hline",
    "    \\end{tabular}",
    "    \\vspace{0.5em}",
    "    \\begin{tabular}{c}",
    "        $^{1}$ Total description-length residual, $^{2}$ Total description-length parameter term",
    "    \\end{tabular}",
    "    \\caption{Top 10 merged selected functions ranked by total description length.}",
    "    \\label{tab:bestfittingCLASH}",
    "\\end{table*}",
]

latex_code = "\n".join(lines)
latex_output.write_text(latex_code)
print(f"Saved LaTeX table: {latex_output}")
print(latex_code)